In [ ]:
"""
Batch CDSE download + training tiles pipeline.

What it does:
- For each AOI + date window:
  - Search Sentinel-2 L2A products (cloud filtered)
  - For each S2 product, find nearest-in-time Sentinel-1 GRD product over same AOI
  - Download both SAFE.zip files (streaming)
  - Run your existing build_training_tiles(S1_ZIP, S2_ZIP) to produce NPZ tiles

Uses:
- CDSE access token (password grant) per CDSE docs.  :contentReference[oaicite:1]{index=1}
- CDSE OData Products endpoint to search + download via /$value. :contentReference[oaicite:2]{index=2}

Env vars required:
  CDSE_USER, CDSE_PASS
Optional:
  CDSE_TOTP  (if you have 2FA)
"""

from dotenv import load_dotenv
import os, json, zipfile, subprocess, time
from dataclasses import dataclass
from pathlib import Path
from datetime import datetime, timezone
from typing import List, Tuple, Dict, Any, Optional

import requests
import numpy as np
import rasterio
from rasterio.windows import Window
from rasterio.warp import reproject, Resampling
from rasterio.transform import xy
from pyproj import Transformer

# -----------------------------
# Your existing pipeline settings
# -----------------------------
RAW_DIR   = Path("data/raw")
PROC_DIR  = Path("data/proc")
OUT_DIR   = Path("data/tiles_npz")

SNAP_GPT = Path.home() / "esa-snap" / "bin" / "gpt"
GRAPH_TC = Path("s1_grd_to_tc_dim.xml")  # your graph (already includes speckle filter)

TILE   = 256
STRIDE = 256
MIN_VALID_FRAC = 0.8
SCL_INVALID = {3, 8, 9, 10, 11}
#SCL_INVALID = {1, 2, 3, 7, 8, 9, 10, 11}
OCEAN_STD_THR = 1.2
S2_BANDS_13 = ["B01","B02","B03","B04","B05","B06","B07","B8A","B09","B11","B12"] # Note that B08 and B10 is missing since it is not included in data


RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# CDSE endpoints
# -----------------------------
TOKEN_URL = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
ODATA_ROOT = "https://catalogue.dataspace.copernicus.eu/odata/v1"

# -----------------------------
# Batch search config (edit)
# -----------------------------
@dataclass
class Job:
    name: str
    bbox_lonlat: Tuple[float, float, float, float]  # (minLon, minLat, maxLon, maxLat)
    date_start: str  # "YYYY-MM-DD"
    date_end: str    # "YYYY-MM-DD"
    max_s2: int = 3
    max_cloud: float = 20.0
    max_time_diff_hours: int = 36  # max allowed S1-S2 time gap

JOBS: List[Job] = [
    Job(
        name="dk_test",
        bbox_lonlat=(9.8, 54.5, 12.8, 57.8),   # example bbox
        date_start="2025-08-17",
        date_end="2025-08-18",
        max_s2=999,
        max_cloud=30.0,
    ),
    # add more AOIs / date windows here
    # Job(
    #     name="benelux_nw_germany_20tiles",
    #     bbox_lonlat=(2.0, 49.0, 10.5, 53.8),
    #     date_start="2025-08-15",
    #     date_end="2025-08-25",
    #     max_s2=20,
    #     max_cloud=20.0,   # relax if you want to reliably get 20
    #     max_time_diff_hours=36,
    # ),
]

DOWNLOAD_DIR = Path("data/downloads")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Helpers: CDSE auth + OData search/download
# -----------------------------
load_dotenv()  # loads .env from current working directory by default

def cdse_token() -> str:
    user = os.environ.get("CDSE_USER")
    pw = os.environ.get("CDSE_PASS")
    totp = os.environ.get("CDSE_TOTP")

    if not user or not pw:
        raise RuntimeError("Set env vars CDSE_USER and CDSE_PASS")

    data = {
        "client_id": "cdse-public",
        "grant_type": "password",
        "username": user,
        "password": pw,
    }
    if totp:
        data["totp"] = totp

    r = requests.post(TOKEN_URL, data=data, timeout=60)
    r.raise_for_status()
    return r.json()["access_token"]

def odata_get(url, token, params=None, timeout=120, retries=1):
    def _do(tok):
        return requests.get(url, headers={"Authorization": f"Bearer {tok}"}, params=params, timeout=timeout)

    r = _do(token)
    if r.status_code in (401, 403, 429) and retries > 0:
        # refresh token and retry once
        token = cdse_token()
        r = _do(token)
    r.raise_for_status()
    return r, token

def scene_clear_fraction(scl_on_s2_10m: Path, b02_10m: Path, step=512) -> float:
    # sample both SCL and B02 sparsely to keep it fast
    with rasterio.open(scl_on_s2_10m) as ds_scl, rasterio.open(b02_10m) as ds_b02:
        scl = ds_scl.read(1)[::step, ::step].astype(np.uint8)
        b02 = ds_b02.read(1)[::step, ::step]

    # only evaluate “clear” where S2 actually has data
    has_data = b02 > 0
    if has_data.sum() == 0:
        return 0.0  # all no-data, treat as not usable

    clear = ~np.isin(scl, list(SCL_INVALID))
    return float((clear & has_data).sum() / has_data.sum())

def warp_band_to_ref_float32(src_path: Path, ref_path: Path, out_path: Path, resampling=Resampling.bilinear):
    with rasterio.open(ref_path) as ref, rasterio.open(src_path) as src:
        prof = ref.profile.copy()
        prof.update(driver="GTiff", count=1, dtype="float32", compress="deflate",
                    predictor=2, tiled=True, blockxsize=256, blockysize=256)

        dst = np.zeros((ref.height, ref.width), dtype=np.float32)

        reproject(
            source=rasterio.band(src, 1),
            destination=dst,
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=ref.transform, dst_crs=ref.crs,
            resampling=resampling,
        )

        out_path.parent.mkdir(parents=True, exist_ok=True)
        with rasterio.open(out_path, "w", **prof) as out:
            out.write(dst, 1)

def find_s2_band_paths(s2_safe: Path):
    r10 = (list(s2_safe.glob("**/IMG_DATA/R10m")))[0]
    r20 = (list(s2_safe.glob("**/IMG_DATA/R20m")))[0]
    r60 = (list(s2_safe.glob("**/IMG_DATA/R60m")))[0]

    p = {}
    # 10m
    p["B02"] = list(r10.glob("*_B02_10m.jp2"))[0]
    p["B03"] = list(r10.glob("*_B03_10m.jp2"))[0]
    p["B04"] = list(r10.glob("*_B04_10m.jp2"))[0]

    # 20m
    for b in ["B05","B06","B07","B8A","B11","B12"]:
        p[b] = list(r20.glob(f"*_ {b}_20m.jp2".replace(" ", "")))[0]

    # 60m
    for b in ["B01","B09"]:
        p[b] = list(r60.glob(f"*_ {b}_60m.jp2".replace(" ", "")))[0]

    # SCL (still needed for clouds)
    p["SCL"] = list(r20.glob("*_SCL_20m.jp2"))[0]

    return p

def prepare_s2_allbands_10m(s2_zip: Path, scene_id: str):
    s2_unzip_dir = RAW_DIR / s2_zip.stem.replace(".SAFE", "")
    s2_safe = unzip_safe(s2_zip, s2_unzip_dir)

    paths = find_s2_band_paths(s2_safe)
    ref_10m = paths["B02"]

    warped = {}
    for b in S2_BANDS_13:
        out = PROC_DIR / f"{scene_id}__{b}_on10m.tif"
        if not out.exists():
            warp_band_to_ref_float32(paths[b], ref_10m, out, resampling=Resampling.bilinear)
        warped[b] = out

    # SCL needs nearest (classes)
    scl_on10m = PROC_DIR / f"{scene_id}__SCL_on10m.tif"
    if not scl_on10m.exists():
        warp_scl_to_ref_uint8(paths["SCL"], ref_10m, scl_on10m)

    return warped, scl_on10m, ref_10m

def read_s2_stack_13(warped_band_paths: dict, window: Window):
    arrs = []
    for b in S2_BANDS_13:
        with rasterio.open(warped_band_paths[b]) as ds:
            arrs.append(ds.read(1, window=window))
    return np.stack(arrs, axis=0)


def bbox_to_wkt(bbox_lonlat):
    minx, miny, maxx, maxy = bbox_lonlat
    # POLYGON((lon lat,...))
    return (
        "POLYGON(("
        f"{minx} {miny},"
        f"{maxx} {miny},"
        f"{maxx} {maxy},"
        f"{minx} {maxy},"
        f"{minx} {miny}"
        "))"
    )

def odata_search_s2(token: str, bbox_lonlat, date_start: str, date_end: str,
                    max_cloud: float, top: int) -> Tuple[List[Dict[str, Any]], str]:
    """
    Search Sentinel-2 L2A products via OData.
    Returns list of product dicts (contains Id, Name, ContentDate, etc).
    """
    wkt = bbox_to_wkt(bbox_lonlat)
    # OData filter:
    # - Collection == SENTINEL-2
    # - product type L2A
    # - intersects AOI
    # - time window
    # - cloud cover
    # Note: attributes naming can differ by collection; this pattern is commonly used in CDSE examples. :contentReference[oaicite:3]{index=3}
    dt0 = f"{date_start}T00:00:00.000Z"
    dt1 = f"{date_end}T23:59:59.999Z"

    filt = " and ".join([
        "Collection/Name eq 'SENTINEL-2'",
        "contains(Name,'MSIL2A')",
        f"ContentDate/Start gt {dt0}",
        f"ContentDate/Start lt {dt1}",
        f"OData.CSC.Intersects(area=geography'SRID=4326;{wkt}')",
        f"Attributes/OData.CSC.DoubleAttribute/any(a: a/Name eq 'cloudCover' and a/OData.CSC.DoubleAttribute/Value le {max_cloud})",
    ])

    url = f"{ODATA_ROOT}/Products"
    params = {
        "$filter": filt,
        "$top": str(top),
        "$orderby": "ContentDate/Start desc",
    }
    r, token = odata_get(url, token, params=params, timeout=120, retries=1)
    return r.json().get("value", []), token

def odata_search_s1(token: str, bbox_lonlat, dt_center_iso: str, hours: int = 48, top: int = 5) -> Tuple[List[Dict[str, Any]], str]:
    """
    Search Sentinel-1 GRD around a center time (±hours).
    """
    wkt = bbox_to_wkt(bbox_lonlat)
    center = datetime.fromisoformat(dt_center_iso.replace("Z", "+00:00"))
    dt0 = (center - __import__("datetime").timedelta(hours=hours)).strftime("%Y-%m-%dT%H:%M:%S.000Z")
    dt1 = (center + __import__("datetime").timedelta(hours=hours)).strftime("%Y-%m-%dT%H:%M:%S.000Z")

    filt = " and ".join([
        "Collection/Name eq 'SENTINEL-1'",
        "contains(Name,'IW_GRDH')",
        "contains(Name,'1SDV')",  # VV/VH
        f"ContentDate/Start gt {dt0}",
        f"ContentDate/Start lt {dt1}",
        f"OData.CSC.Intersects(area=geography'SRID=4326;{wkt}')",
    ])

    url = f"{ODATA_ROOT}/Products"
    params = {"$filter": filt, "$top": str(top), "$orderby": "ContentDate/Start desc"}
    r, token = odata_get(url, token, params=params, timeout=120, retries=1)
    return r.json().get("value", []), token

def parse_dt(prod: dict) -> datetime:
    # ContentDate: { "Start": "...Z", "End": "...Z" }
    s = prod["ContentDate"]["Start"]
    return datetime.fromisoformat(s.replace("Z", "+00:00"))

def pick_nearest_by_time(candidates: List[dict], target_dt: datetime) -> Optional[dict]:
    if not candidates:
        return None
    best = min(candidates, key=lambda p: abs((parse_dt(p) - target_dt).total_seconds()))
    return best

def download_product_zip(get_token_fn, token: str, prod: dict, out_dir: Path) -> tuple[Path, str]:
    """
    Download product SAFE.zip via OData /$value with redirect-safe auth.
    Returns: (zip_path, possibly_updated_token)
    """
    pid = prod["Id"]
    name = prod["Name"]
    out = out_dir / f"{name}.zip" if name.endswith(".SAFE") else out_dir / f"{name}.SAFE.zip"

    # already downloaded
    if out.exists() and out.stat().st_size > 10_000_000:
        return out, token

    url = f"{ODATA_ROOT}/Products({pid})/$value"
    out_tmp = out.with_suffix(".SAFE.zip.part")

    def _stream_download(url_to_get: str, bearer: str):
        headers = {"Authorization": f"Bearer {bearer}"}
        with requests.get(url_to_get, headers=headers, stream=True, timeout=300, allow_redirects=False) as r:
            # Manual redirect handling so Authorization header is preserved
            if r.status_code in (301, 302, 303, 307, 308):
                loc = r.headers.get("Location")
                if not loc:
                    r.raise_for_status()
                with requests.get(loc, headers=headers, stream=True, timeout=300) as r2:
                    r2.raise_for_status()
                    with open(out_tmp, "wb") as f:
                        for chunk in r2.iter_content(chunk_size=1024 * 1024):
                            if chunk:
                                f.write(chunk)
                return

            r.raise_for_status()
            with open(out_tmp, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)

    # Try with current token; if auth fails, refresh token and retry once
    try:
        _stream_download(url, token)
    except requests.HTTPError as e:
        status = getattr(e.response, "status_code", None)
        if status in (401, 403):
            token = get_token_fn()
            _stream_download(url, token)
        else:
            raise

    out_tmp.rename(out)
    return out, token

# -----------------------------
# Your existing pipeline code (slightly compacted)
# -----------------------------
def run(cmd):
    cmd = list(map(str, cmd))
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

def unzip_safe(zip_path: Path, out_dir: Path) -> Path:
    """
    Unzip SAFE.zip into out_dir, but skip if a .SAFE folder already exists.
    Returns the .SAFE folder path.
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    # If already unzipped, reuse it
    safes = list(out_dir.glob("*.SAFE")) or list(out_dir.glob("**/*.SAFE"))
    if safes:
        return safes[0]

    # Otherwise unzip once
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(out_dir)

    safes = list(out_dir.glob("*.SAFE")) or list(out_dir.glob("**/*.SAFE"))
    if not safes:
        raise FileNotFoundError(f"No .SAFE found after unzip: {zip_path}")
    return safes[0]

def warp_scl_to_ref_uint8(src_path: Path, ref_path: Path, out_path: Path):
    with rasterio.open(ref_path) as ref, rasterio.open(src_path) as src:
        prof = ref.profile.copy()
        prof.update(driver="GTiff", count=1, dtype="uint8", compress="deflate",
                    tiled=True, blockxsize=256, blockysize=256)
        dst = np.zeros((ref.height, ref.width), dtype=np.uint8)
        reproject(
            source=rasterio.band(src, 1),
            destination=dst,
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=ref.transform, dst_crs=ref.crs,
            resampling=Resampling.nearest,
        )
        out_path.parent.mkdir(parents=True, exist_ok=True)
        with rasterio.open(out_path, "w", **prof) as out:
            out.write(dst, 1)

def warp_s1_imgs_to_s2_grid(vv_path: Path, vh_path: Path, ref_10m: Path, out_path: Path):
    with rasterio.open(ref_10m) as ref, rasterio.open(vv_path) as vv, rasterio.open(vh_path) as vh:
        prof = ref.profile.copy()
        prof.update(driver="GTiff", count=2, dtype="float32", compress="deflate",
                    predictor=2, tiled=True, blockxsize=256, blockysize=256)
        dst = np.zeros((2, ref.height, ref.width), dtype=np.float32)

        reproject(
            source=rasterio.band(vv, 1),
            destination=dst[0],
            src_transform=vv.transform, src_crs=vv.crs,
            dst_transform=ref.transform, dst_crs=ref.crs,
            resampling=Resampling.bilinear,
        )
        reproject(
            source=rasterio.band(vh, 1),
            destination=dst[1],
            src_transform=vh.transform, src_crs=vh.crs,
            dst_transform=ref.transform, dst_crs=ref.crs,
            resampling=Resampling.bilinear,
        )

        # linear sigma0 -> dB
        dst = 10.0 * np.log10(np.maximum(dst, 1e-10)).astype(np.float32)

        out_path.parent.mkdir(parents=True, exist_ok=True)
        with rasterio.open(out_path, "w", **prof) as out:
            out.write(dst)

def scl_window_mask(scl_on_s2_10m: Path, window: Window) -> np.ndarray:
    with rasterio.open(scl_on_s2_10m) as ds:
        scl = ds.read(1, window=window).astype(np.uint8)
    return ~np.isin(scl, list(SCL_INVALID))

def epsg_from_s2_b02(b02_10m: Path) -> str:
    with rasterio.open(b02_10m) as ds:
        epsg = ds.crs.to_epsg()
        if epsg is None:
            raise ValueError(f"Could not determine EPSG from {b02_10m} (CRS={ds.crs})")
        return f"EPSG:{epsg}"

def process_s1_to_tc_imgs(s1_zip: Path, map_proj: str):
    if not SNAP_GPT.exists():
        raise FileNotFoundError(f"SNAP GPT not found: {SNAP_GPT}")
    if not GRAPH_TC.exists():
        raise FileNotFoundError(f"GRAPH_TC not found: {GRAPH_TC}")

    s1_unzip_dir = RAW_DIR / s1_zip.stem.replace(".SAFE", "")
    s1_safe = unzip_safe(s1_zip, s1_unzip_dir)

    base = s1_zip.stem.replace(".SAFE", "")
    s1_tc_dim = PROC_DIR / (base + "_tc.dim")

    if not s1_tc_dim.exists():
        # assumes your XML uses <mapProjection>${mapProjection}</mapProjection>
        run([SNAP_GPT, GRAPH_TC, f"-Pin={s1_safe}", f"-Pout={s1_tc_dim}", f"-PmapProjection={map_proj}"])

    tc_data = PROC_DIR / (base + "_tc.data")
    vv = tc_data / "Sigma0_VV.img"
    vh = tc_data / "Sigma0_VH.img"
    if not vv.exists() or not vh.exists():
        raise FileNotFoundError(f"Missing VV/VH outputs in {tc_data}")
    return vv, vh

def tile_and_write(warped_s2: dict, s1_on_s2: Path, scl_on10m: Path, b02_10m: Path, scene_id: str):
    with rasterio.open(b02_10m) as ref, rasterio.open(s1_on_s2) as s1ds:
        H, W = ref.height, ref.width
        wrote = skipped_valid = skipped_ocean = 0

        for r0 in range(0, H - TILE + 1, STRIDE):
            for c0 in range(0, W - TILE + 1, STRIDE):
                win = Window(c0, r0, TILE, TILE)

                s2 = read_s2_stack_13(warped_s2, win)
                s1 = s1ds.read([1,2], window=win).astype(np.float32)

                clear = scl_window_mask(scl_on10m, win)
                
                b02 = s2[S2_BANDS_13.index("B02")]
                nonzero = b02 > 0

                # Optional (strongly recommended): require S1 coverage (avoid outside footprint)
                s1_ok = np.isfinite(s1[0]) & np.isfinite(s1[1]) & (s1[0] > -80) & (s1[1] > -80)

                valid = clear & nonzero & s1_ok
                valid_frac = float(valid.mean())
                if valid_frac < MIN_VALID_FRAC:
                    skipped_valid += 1
                    continue

                vv_db = s1[0]
                score = float(np.nanstd(vv_db[valid]))
                if score < OCEAN_STD_THR:
                    skipped_ocean += 1
                    continue

                meta = {
                    "scene": scene_id,
                    "row0": int(r0), "col0": int(c0),
                    "tile": int(TILE), "stride": int(STRIDE),
                    "valid_frac": valid_frac,
                    "vv_db_std": score,
                    "crs": str(ref.crs),
                }

                out = OUT_DIR / f"{scene_id}_r{r0}_c{c0}.npz"
                np.savez_compressed(out, s1=s1, s2=s2, valid=valid.astype(np.uint8), meta=json.dumps(meta))
                wrote += 1

        print(f"[{scene_id}] wrote={wrote} skipped_valid={skipped_valid} skipped_ocean={skipped_ocean}")

def build_training_tiles(s1_zip: Path, s2_zip: Path):
    scene_id = s2_zip.stem.replace(".SAFE", "")

    # NEW: prepare all 13 S2 bands on 10m grid (and SCL on 10m)
    warped_s2, scl_on10m, b02_10m = prepare_s2_allbands_10m(s2_zip, scene_id)
    map_proj = epsg_from_s2_b02(b02_10m)

    vv_img, vh_img = process_s1_to_tc_imgs(s1_zip, map_proj)

    # Warp S1 to S2 10m grid
    s1_on_s2 = PROC_DIR / f"{scene_id}__S1_on_S2_10m.tif"
    if not s1_on_s2.exists():
        warp_s1_imgs_to_s2_grid(vv_img, vh_img, b02_10m, s1_on_s2)

    # Now tile using the warped 13-band stack + scl_on10m
    tile_and_write(warped_s2, s1_on_s2, scl_on10m, b02_10m, scene_id)


# -----------------------------
# Batch driver
# -----------------------------
def run_batch():
    token = cdse_token()

    for job in JOBS:
        print(f"\n=== JOB: {job.name} {job.date_start}..{job.date_end} bbox={job.bbox_lonlat} ===")

        s2_list, token = odata_search_s2(
            token=token,
            bbox_lonlat=job.bbox_lonlat,
            date_start=job.date_start,
            date_end=job.date_end,
            max_cloud=job.max_cloud,
            top=job.max_s2
        )
        if not s2_list:
            print("No S2 found.")
            continue

        for s2_prod in s2_list:
            s2_dt = parse_dt(s2_prod)
            s1_cands, token = odata_search_s1(
                token=token,
                bbox_lonlat=job.bbox_lonlat,
                dt_center_iso=s2_prod["ContentDate"]["Start"],
                hours=job.max_time_diff_hours,
                top=10
            )
            s1_prod = pick_nearest_by_time(s1_cands, s2_dt)
            if not s1_prod:
                print("No S1 match for", s2_prod["Name"])
                continue

            # Check time gap
            gap_h = abs((parse_dt(s1_prod) - s2_dt).total_seconds()) / 3600.0
            if gap_h > job.max_time_diff_hours:
                print(f"Skip pair (time gap {gap_h:.1f}h) S2={s2_prod['Name']} S1={s1_prod['Name']}")
                continue

            print(f"PAIR: S2={s2_prod['Name']}  S1={s1_prod['Name']}  gap={gap_h:.1f}h")

            s2_zip, token = download_product_zip(cdse_token, token, s2_prod, DOWNLOAD_DIR)
            s1_zip, token = download_product_zip(cdse_token, token, s1_prod, DOWNLOAD_DIR)

            # Run your tile builder
            build_training_tiles(s1_zip, s2_zip)

if __name__ == "__main__":
    run_batch()



=== JOB: dk_test 2025-08-17..2025-08-18 bbox=(9.8, 54.5, 12.8, 57.8) ===
PAIR: S2=S2A_MSIL2A_20250818T103041_N0511_R108_T32VPH_20250818T143613.SAFE  S1=S1C_IW_GRDH_1SDV_20250819T053147_20250819T053212_003737_007770_345A_COG.SAFE  gap=19.0h
Running: /home/jakob/esa-snap/bin/gpt s1_grd_to_tc_dim.xml -Pin=data/raw/S1C_IW_GRDH_1SDV_20250819T053147_20250819T053212_003737_007770_345A_COG/S1C_IW_GRDH_1SDV_20250819T053147_20250819T053212_003737_007770_345A_COG.SAFE -Pout=data/proc/S1C_IW_GRDH_1SDV_20250819T053147_20250819T053212_003737_007770_345A_COG_tc.dim -PmapProjection=EPSG:32632


INFO: org.esa.snap.core.gpf.operators.tooladapter.ToolAdapterIO: Initializing external tool adapters
INFO: org.esa.snap.core.util.EngineVersionCheckActivator: Please check regularly for new updates for the best SNAP experience.


Executing processing graph


[main] INFO hdf.hdflib.HDFLibrary - HDF4 library: 
[main] INFO hdf.hdflib.HDFLibrary -  successfully loaded.
[main] INFO hdf.hdf5lib.H5 - HDF5 library: 
[main] INFO hdf.hdf5lib.H5 -  successfully loaded.


....10%....20%....30%....40%....50%....60%..